# Swan — Data check for the questionnaire

## MVP scope decided on 23/10/2026

- **Concerne: **acne concern**
- **Output for MVP:** Moisturizers

## 📦 Data sources

| Source | File | Downloaded | Used here? |
|---|---|---|---|
| Open Beauty Facts | `raw_data/fr.openbeautyfacts.org.products.csv` | 25/10/2026 | ✅ Yes |
| EU Cosmetics Regulation (EC) 1223/2009 — consolidated 18.05.2026, Annex VI (UV filters) | `raw_data/eu_uv_filters_annex_vi.csv` (extracted from the PDF, ref + page per row) | 25/10/2026 | ✅ Yes |
| EU Cosmetics Regulation (EC) 1223/2009 — consolidated 18.05.2026, Annex III (fragrance allergens) | `raw_data/eu_fragrance_allergens_annex_iii.csv` (extracted from the PDF, ref + page per row) | 25/10/2026 | ✅ Yes |
| Open Beauty Facts — Parquet on Hugging Face (`openfoodfacts/product-database`, `beauty.parquet`, 64.4 MB) | — | — | ❌ Not used — alternative source to discuss with Lina |
| EU glossary of common ingredient names — [Decision (EU) 2025/1175](https://eur-lex.europa.eu/legal-content/EN/TXT/HTML/?uri=OJ:L_202501175) | — | — | ❌ No (useful for J1, INCI normalisation) |
| CosIng — EU cosmetic ingredient database | — | — | ❌ Not yet (J1, tasks T09 / T11) |


## 🎯  Questionnaire for MVP 
| # | Question | Profile field | Options shown → stored value | Required | Action | Based on |
|---|---|---|---|---|---|---|
| 1 | Are you 18 or older? | `is_adult` | yes → `true` / no → `false` | ✅ | refer if `false` | scope rule |
| 2 | What would you like help with? | `concern` | a simple routine for acne-prone skin → `acne_prone` / something else → `other` | ✅ | out-of-scope message if `other` | MVP scope  |
| 3 | How would you describe your blemishes? | `blemish_level` | rank from 1 to 5 | ✅ |  | referral rule | Validated on 26/10
| 4 | Does any of these apply to you? | `referral_flags` | prescribed treatment → `prescribed_treatment` / sudden or unusual lesion → `unusual_lesion` / pregnant or breastfeeding → `pregnant_or_breastfeeding` / none of these → empty list | ✅ | refer if not empty | referral rule |
| 5 | What is your skin type? | `skin_type` | `oily` / `combination` / `normal` / `dry` / not sure → `not_sure` | ✅ | rank | data to review |
| 6 | Does your skin react easily (stinging, redness)? | `reactivity` | `never` / `sometimes` / `often` | ✅ | rank (penalise fragrance) | data to review | GO with Yes or No for the MVP
| 7 | Ingredients you prefer to avoid? | `avoid` | fragrance → `fragrance` / essential oils → `essential_oils` / none → empty list | ❌ | filter 
| 8 | Which texture do you prefer? | `texture` | light → `light` / rich → `rich` / no preference → `no_preference` | ❌ (default `no_preference`) | NOT FOR THE MVP 


## To validate on 26/10

| # | Decision | Proposal | Decision |
|---|---|---|---|
| B02 | MVP | Adults with acne-prone skin, mild blemishes · cleanser + moisturizer + facial sunscreen · Number of products we need in each category (cleanser, moisturizer, sunscreen).| 1 issue acne - output moisturizer
| B04 | Market scope | EU (EU + EEA as an option: the regulation has "EEA relevance") · France is the core of the data | Start with France as a start
| — | Role edge cases | SPF moisturizer stays a moisturizer (sunscreen slot = dedicated sunscreens) · toners out · micellar water = cleanser · face lotions after examples · anti-ageing out | To split into to different categories moisterizer and sunscreen based on the tags.
| T12 | Questionnaire v1 + profile schema | 8 questions, safety first 
| T23 | Test-profile grid | Built from the profile fields; first 6 profiles in section 11
| T27 | Mockups | 4 screens: welcome, questionnaire, routine, referral 
| T06 | Data source | Same snapshot for the whole team (this CSV or `beauty.parquet`)

**File facts** :
- Number of lines : 64 238 
- About 200 columns, most of them food nutrition columns (`*_100g`) inherited from Open Food Facts


## Settings to run my notebook

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [27]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

FILE_PATH = "../raw_data/en.openbeautyfacts.org.products.csv"
UV_FILTERS_PATH = "../raw_data/eu_uv_filters_annex_vi.csv"
ALLERGENS_PATH = "../raw_data/eu_fragrance_allergens_annex_iii.csv"
COSING_PATH = "../raw_data/Swan_CosIng_Ingredient_Reference_2026-10-26.csv"
LINES_IN_FILE = 64238    # from the terminal: wc -l

SEED = 42                # random_state: same random sample for everyone
NB_ROUTINE_PRODUCT = 1   # moisturizer candidates needed per profile (MVP output = moisturizers, B02)
BLEMISH_REFERRAL_LEVEL = 5 # Q3: from this level on the 1–5 scale

EU_COUNTRY_TAGS = [
    "en:france", "en:belgium", "en:germany", "en:spain", "en:italy", "en:netherlands",
    "en:luxembourg", "en:portugal", "en:austria", "en:ireland", "en:greece", "en:poland",
    "en:czech-republic", "en:czechia", "en:slovakia", "en:hungary", "en:romania", "en:bulgaria",
    "en:croatia", "en:slovenia", "en:denmark", "en:sweden", "en:finland", "en:estonia",
    "en:latvia", "en:lithuania", "en:cyprus", "en:malta",
]
EEA_EXTRA_TAGS = ["en:norway", "en:iceland", "en:liechtenstein"]
COUNTRY_TAGS = ["en:france"]   # B04 (26/10): France to start — later: EU_COUNTRY_TAGS, then + EEA_EXTRA_TAGS

print(FILE_PATH)

../raw_data/en.openbeautyfacts.org.products.csv


## Load Data Set and check the columns

In [28]:
df_complete_file = pd.read_csv(FILE_PATH, sep="\t", dtype={"code": str})
df_complete_file.shape

/var/folders/bl/bvf3l2w16sb0jmy2v926fp0h0000gn/T/ipykernel_72675/1418356569.py:1: DtypeWarning: Columns (11,34,36,54,60) have mixed types. Specify dtype option on import or set low_memory=False.
  df_complete_file = pd.read_csv(FILE_PATH, sep="\t", dtype={"code": str})


(64237, 200)

In [29]:
df_complete_file.head()

,code,url,creator,created_t,created_datetime,last_modified_t,last_modified_datetime,last_modified_by,last_updated_t,last_updated_datetime,...,choline_100g,phylloquinone_100g,beta-glucan_100g,inositol_100g,carnitine_100g,sulphate_100g,nitrate_100g,acidity_100g,carbohydrates-total_100g,water_100g
0,00000001,http://world-en.openbeautyfacts.org/product/00...,tonyracket,1751609682,2025-07-04T06:14:42Z,1751609686,2025-07-04T06:14:46Z,tonyracket,1.751610e+09,2025-07-04T06:14:46Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,00000014,http://world-en.openbeautyfacts.org/product/00...,insectproductadd,1710345659,2024-03-13T16:00:59Z,1748277440,2025-05-26T16:37:20Z,fitcounter,1.748277e+09,2025-05-26T16:37:20Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00000092,http://world-en.openbeautyfacts.org/product/00...,product-scan-com,1638580447,2021-12-04T01:14:07Z,1744901851,2025-04-17T14:57:31Z,smoothie-app,1.744902e+09,2025-04-17T14:57:31Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,00000163,http://world-en.openbeautyfacts.org/product/00...,smoothie-app,1682257711,2023-04-23T13:48:31Z,1727959822,2024-10-03T12:50:22Z,fix-code-bot,1.743167e+09,2025-03-28T13:01:25Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,00000280,http://world-en.openbeautyfacts.org/product/00...,date-limite-app,1584379762,2020-03-16T17:29:22Z,1744884567,2025-04-17T10:09:27Z,smoothie-app,1.744885e+09,2025-04-17T10:09:27Z,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
df_complete_file["completeness"].describe()

count    56361.000000
mean         0.310339
std          0.220251
min          0.050000
25%          0.075000
50%          0.275000
75%          0.462500
max          1.100000
Name: completeness, dtype: float64

In [31]:
# products created per year
df_complete_file["created_datetime"].fillna("").str[:4].value_counts().sort_index()

created_datetime
2013        6
2014      148
2015      244
2016     2556
2017     1736
2018     5153
2019     3953
2020     4359
2021     5865
2022     4967
2023     5943
2024     8437
2025    16391
2026     4479
Name: count, dtype: int64

In [32]:
df_complete_file.dtypes.value_counts()

float64    132
object      66
int64        2
Name: count, dtype: int64

In [33]:
print("Lines in the file (wc -l) :", LINES_IN_FILE)
print("Rows loaded by pandas     :", len(df_complete_file))
print("Gap                       :", LINES_IN_FILE - 1 - len(df_complete_file))
print("Duplicate codes           :", df_complete_file["code"].duplicated().sum())
print("Last product modified     :", df_complete_file["last_modified_datetime"].max())

Lines in the file (wc -l) : 64238
Rows loaded by pandas     : 64237
Gap                       : 0
Duplicate codes           : 0
Last product modified     : 2026-05-07T23:52:37Z


## Do we have all the fields needed complete?

In [34]:
states = df_complete_file["states_tags"].fillna("")
for part in ["ingredients", "categories", "product-name", "brands"]:
    done = states.str.contains("en:" + part + "-completed").sum()
    print(f"{part:14} completed: {done:6} ({round(done / len(df_complete_file) * 100, 1)} %)")

ingredients    completed:  17303 (26.9 %)
categories     completed:  28496 (44.4 %)
product-name   completed:  40029 (62.3 %)
brands         completed:  34549 (53.8 %)


In [35]:
text = df_complete_file["ingredients_text"].fillna("").str.strip()
print("Ingredient text not empty :", (text != "").sum())
print("Ingredient text ≥ 20 chars:", (text.str.len() >= 20).sum())
print("Marked 'completed'        :", states.str.contains("en:ingredients-completed").sum())

Ingredient text not empty : 17625
Ingredient text ≥ 20 chars: 17051
Marked 'completed'        : 17303


In [8]:
states = df_complete_file["states_tags"].fillna("")
for part in ["ingredients", "categories", "product-name", "brands"]:
    done = states.str.contains("en:" + part + "-completed").sum()
    print(f"{part:14} completed: {done:6} ({round(done / len(df_complete_file) * 100, 1)} %)")

ingredients    completed:  17303 (26.9 %)
categories     completed:  28496 (44.4 %)
product-name   completed:  40029 (62.3 %)
brands         completed:  34549 (53.8 %)


In [9]:
#Check if we have all EU countries in the dataset
countries = df_complete_file["countries_tags"].fillna("")
single_country = countries[(countries != "") & ~countries.str.contains(",")]
single_country.value_counts().head(40)

countries_tags
en:france                  19893
en:germany                  4620
en:united-states            3947
en:morocco                  3782
en:spain                    2961
en:united-kingdom           2079
en:italy                    1436
en:india                    1399
en:netherlands              1391
en:portugal                  958
en:canada                    953
en:norway                    926
en:russia                    805
en:switzerland               683
en:belgium                   662
en:algeria                   607
en:saudi-arabia              600
en:romania                   543
en:brazil                    441
en:poland                    376
en:ukraine                   348
en:sweden                    347
en:turkey                    346
en:argentina                 346
en:austria                   315
en:philippines               313
en:mexico                    310
en:australia                 306
en:united-arab-emirates      306
en:colombia                 

In [10]:
# which countries appear most?
for tag in EU_COUNTRY_TAGS + ["en:united-kingdom", "en:switzerland", "en:united-states"]:
    n = countries.str.contains(tag).sum()
    if n > 0:
        print(f"{tag:22} {n}")

en:france              21780
en:belgium             762
en:germany             5017
en:spain               3224
en:italy               1519
en:netherlands         1452
en:luxembourg          79
en:portugal            990
en:austria             403
en:ireland             187
en:greece              258
en:poland              390
en:czech-republic      118
en:slovakia            62
en:hungary             93
en:romania             588
en:bulgaria            206
en:croatia             107
en:slovenia            34
en:denmark             78
en:sweden              375
en:finland             100
en:estonia             47
en:latvia              52
en:lithuania           127
en:cyprus              60
en:malta               20
en:united-kingdom      2206
en:switzerland         804
en:united-states       4197


## Group columns into families
Group the column into families based on the ones we will use for the questionnaire and for our model.

In [36]:
def column_family(name):
    """Return the family of a column, based on its name."""
    if name.endswith("_100g"):
        return "food nutrition"
    elif "image" in name:
        return "images"
    elif "ingredient" in name:
        return "ingredients"
    elif "categor" in name:
        return "categories"
    elif "label" in name:
        return "labels"
    elif "countr" in name:
        return "countries"
    elif name in ["code", "url", "product_name", "abbreviated_product_name", "generic_name", "quantity"] or "brand" in name:
        return "identity"
    elif "state" in name or "complet" in name or "quality" in name or "created" in name or "modified" in name or "creator" in name:
        return "quality"
    else:
        return "other"

families = pd.Series(df_complete_file.columns).apply(column_family)
families.index = df_complete_file.columns
families.value_counts()

food nutrition    123
other              34
quality            11
identity           10
images              8
categories          5
labels              3
countries           3
ingredients         3
Name: count, dtype: int64

In [12]:
families = pd.Series(df_complete_file.columns).apply(column_family)
families.index = df_complete_file.columns
families.value_counts()

food nutrition    123
other              34
quality            11
identity           10
images              8
categories          5
labels              3
countries           3
ingredients         3
Name: count, dtype: int64

## What should we take main category or category tags ?

In [13]:
# check the categories by iteration
df_complete_file["main_category"].value_counts().head(20)

main_category
en:open-beauty-facts          4271
en:cosmetic-products          3937
en:shampoos                   1051
en:shower-gels                 687
en:deodorants                  505
en:toothpastes                 492
en:sunscreen                   410
en:soaps                       384
en:non-open-products-facts     339
nl:verzorging                  318
en:hand-creams                 261
en:toothpaste                  197
en:lip-balms                   160
en:roll-on-deodorants          153
en:body-milks                  142
en:body-creams                 134
en:anti-dandruff-shampoos      128
en:hair-care                   126
nl:skincare                    125
fr:cremes-douche               123
Name: count, dtype: int64

In [14]:
#check the category tags if this is better to use
total = len(df_complete_file)
generic = df_complete_file["main_category"].isin(["en:open-beauty-facts", "en:cosmetic-products"]).sum()
empty = df_complete_file["main_category"].isnull().sum()
top_20 = df_complete_file["main_category"].value_counts().head(20).sum()

print("All products          :", total)
print("Generic main category :", generic, f"({round(generic / total * 100, 1)} %)")
print("Empty main category   :", empty, f"({round(empty / total * 100, 1)} %)")
print("Top 20 covers         :", top_20, f"({round(top_20 / total * 100, 1)} %)")

All products          : 64237
Generic main category : 8208 (12.8 %)
Empty main category   : 38589 (60.1 %)
Top 20 covers         : 13943 (21.7 %)


In [15]:
# same keywords, two columns: which one finds more of our products?
for word in ["cleans", "moistur", "sunscreen"]:
    in_main = df_complete_file["main_category"].fillna("").str.contains(word, case=False).sum()
    in_tags = df_complete_file["categories_tags"].fillna("").str.contains(word, case=False).sum()
    print(f"{word:10} main_category: {in_main:5}   categories_tags: {in_tags:5}")

cleans     main_category:   127   categories_tags:   277
moistur    main_category:   100   categories_tags:   103
sunscreen  main_category:   430   categories_tags:   467


## Check the products we have in the datase 

In [16]:
products = df_complete_file[df_complete_file["countries_tags"].fillna("").str.contains("|".join(COUNTRY_TAGS))].copy()
print("All products            :", len(df_complete_file))
print("In scope                :", len(products))

products = products[products["ingredients_text"].str.strip().str.len() >= 20].copy()
print("+ usable ingredient list:", len(products))

All products            : 64237
In scope                : 21780
+ usable ingredient list: 7224


In [17]:
total = len(df_complete_file)
generic = df_complete_file["main_category"].isin(["en:open-beauty-facts", "en:cosmetic-products"]).sum()
empty = df_complete_file["main_category"].isnull().sum()
top_20 = df_complete_file["main_category"].value_counts().head(20).sum()

print("All products          :", total)
print("Generic main category :", generic, f"({round(generic / total * 100, 1)} %)")
print("Empty main category   :", empty, f"({round(empty / total * 100, 1)} %)")
print("Top 20 covers         :", top_20, f"({round(top_20 / total * 100, 1)} %)")

All products          : 64237
Generic main category : 8208 (12.8 %)
Empty main category   : 38589 (60.1 %)
Top 20 covers         : 13943 (21.7 %)


In [18]:
# same keywords, two columns: which one finds more of our products?
for word in ["cleans", "moistur", "sunscreen"]:
    in_main = df_complete_file["main_category"].fillna("").str.contains(word, case=False).sum()
    in_tags = df_complete_file["categories_tags"].fillna("").str.contains(word, case=False).sum()
    print(f"{word:10} main_category: {in_main:5}   categories_tags: {in_tags:5}")

cleans     main_category:   127   categories_tags:   277
moistur    main_category:   100   categories_tags:   103
sunscreen  main_category:   430   categories_tags:   467


**Question:** which products are cleansers, moisturizers, sunscreens — and validate the rule I used? 

In [37]:
def find_role(categories):
    """Return the product role from its categories."""
    text = str(categories).lower()
    if "anti-wrinkle" in text or "anti-aging" in text or "anti-age" in text:
        return "other"                                   # ageing concern: outside the MVP
    elif "facial-cream" in text or "face-cream" in text or "cremes-hydratantes" in text or "creme-visage" in text:
        return "moisturizer"                             # clear face moisturizers, before the exclusions
    elif "hair" in text or "shampoo" in text or "body" in text or "hand" in text or "shower" in text or "tooth" in text or "after-sun" in text:
        return "other"
    elif "sunscreen" in text or "sun-protection" in text or "solaire" in text:
        return "sunscreen"
    elif "cleans" in text or "nettoyant" in text:
        return "cleanser"
    elif "moistur" in text or "day-cream" in text:
        return "moisturizer"
    else:
        return "other"

def find_role_from_name(name):
    """Guess the role from words in the product name (English and French)."""
    text = str(name).lower()
    if "spf" in text or "sunscreen" in text or "solaire" in text or "sun cream" in text:
        return "sunscreen"
    elif "cleanser" in text or "nettoyant" in text or "micellar" in text or "micellaire" in text or "face wash" in text:
        return "cleanser"
    elif "moisturi" in text or "hydratant" in text or "day cream" in text or "crème visage" in text or "creme visage" in text:
        return "moisturizer"
    else:
        return "other"

products["role"] = products["categories_tags"].apply(find_role)
products["name_role"] = products["product_name"].apply(find_role_from_name)
pd.crosstab(products["role"], products["name_role"])

name_role,cleanser,moisturizer,other,sunscreen
role,,,,
cleanser,46,3,91,0
moisturizer,4,39,99,0
other,141,221,6367,90
sunscreen,0,4,45,74


**Check** check the name based on the product name

In [20]:
def find_role_from_name(name):
    """Guess the role from words in the product name (English and French)."""
    text = str(name).lower()
    if "spf" in text or "sunscreen" in text or "solaire" in text or "sun cream" in text:
        return "sunscreen"
    elif "cleanser" in text or "nettoyant" in text or "micellar" in text or "micellaire" in text or "face wash" in text:
        return "cleanser"
    elif "moisturi" in text or "hydratant" in text or "day cream" in text or "crème visage" in text or "creme visage" in text:
        return "moisturizer"
    else:
        return "other"

products["name_role"] = products["product_name"].apply(find_role_from_name)
pd.crosstab(products["role"], products["name_role"])

name_role,cleanser,moisturizer,other,sunscreen
role,,,,
cleanser,46,3,91,0
moisturizer,4,39,99,0
other,141,221,6367,90
sunscreen,0,4,45,74


In [38]:
others = products[products["role"] == "other"]
face_words = "face|facial|visage|skin-care|creme|cream|soin"
face_in_other = others[others["categories_tags"].fillna("").str.contains(face_words, case=False)]

print("'Other' products with face-care words:", len(face_in_other))
face_in_other["main_category"].value_counts().head(25)

'Other' products with face-care words: 671


main_category
fr:cremes-douche                        96
en:hand-creams                          65
en:lip-cosmetics                        43
en:face-masks                           29
en:open-beauty-facts                    22
ar:كريمات-مرطبة                         17
en:face                                 17
en:body-creams                          17
en:anti-wrinkles-creams                 16
en:face-lotions                         14
fr:baumes-et-beurres-pour-le-corps      14
en:face-scrubs                          11
en:foundations                          11
fr:cremes-pour-les-pieds                10
en:lip-balms                             9
en:shower-gels                           9
en:day-creams                            8
fr:contours-des-yeux                     8
en:soaps                                 7
fr:poudres-libres                        7
en:night-creams                          7
fr:soin-pour-bebe                        6
fr:produits-marketes-pour-les-hommes    

In [39]:
face_words = "face|facial|visage|skin-care|creme|cream|soin"
others = products[products["role"] == "other"]
face_others = others[others["categories_tags"].fillna("").str.contains(face_words, case=False)]
print("'Other' products with face-care words:", len(face_others))

miss_check = face_others.sample(min(20, len(face_others)), random_state=SEED)
for name, url in zip(miss_check["product_name"], miss_check["url"]):
    print(name, "|", url)

'Other' products with face-care words: 671
Rénovateur Anti-Taches Nuit | http://world-en.openbeautyfacts.org/product/3600541295742/renovateur-anti-taches-nuit-l-oreal
Eau pure H2O | http://world-en.openbeautyfacts.org/product/3286011059610/eau-pure-h2o-biolane
Gommage douceur visage Grain de riz et argan | http://world-en.openbeautyfacts.org/product/3700446411701/gommage-douceur-visage-grain-de-riz-et-argan-belle-bio
Protection surgras douche soin nourrissante | http://world-en.openbeautyfacts.org/product/8712561607391/protection-surgras-douche-soin-nourrissante-unilever
Stick lèvres hydratant | http://world-en.openbeautyfacts.org/product/3560070881253/stick-levres-hydratant-carrefour
Crème de jour ultra riche sans rinçage | http://world-en.openbeautyfacts.org/product/3600541322011/creme-de-jour-ultra-riche-sans-rincage-l-oreal
Visibly Clear correct & perfect - Médium | http://world-en.openbeautyfacts.org/product/3574661239439/visibly-clear-correct-perfect-medium-johnson-johnson
nan | 

In [40]:
products["final_role"] = products["role"]
products.loc[(products["role"] == "other") & (products["name_role"] != "other"), "final_role"] = products["name_role"]
products.loc[products["name_role"] == "cleanser", "final_role"] = "cleanser"
products["final_role"].value_counts()

final_role
other          6367
moisturizer     359
cleanser        285
sunscreen       213
Name: count, dtype: int64

In [41]:
moisturizers = products[products["final_role"] == "moisturizer"]
check = moisturizers.sample(min(10, len(moisturizers)), random_state=SEED)
for name, url in zip(check["product_name"], check["url"]):
    print(name, "|", url)

Intensif 7 jours Lait Hydratant Apaisant au L-Bifidus | http://world-en.openbeautyfacts.org/product/3600541125360/intensif-7-jours-lait-hydratant-apaisant-au-l-bifidus-l-oreal
Lait corps hydratant au beurre de karité bio | http://world-en.openbeautyfacts.org/product/3245678042205/lait-corps-hydratant-au-beurre-de-karite-bio-cosmia-bio
masque hydratant | http://world-en.openbeautyfacts.org/product/3760100682779/masque-hydratant-evoluderm
Crème visage | http://world-en.openbeautyfacts.org/product/3770004051007/creme-visage-royer-cosmetique
Cosmia gel douche et bain lait hydratant a la noix de coco | http://world-en.openbeautyfacts.org/product/3245678044490/cosmia-gel-douche-et-bain-lait-hydratant-a-la-noix-de-coco
Crème Mains Hydratante Pur Arnica | http://world-en.openbeautyfacts.org/product/3660005986524/creme-mains-hydratante-pur-arnica-yves-rocher
Crème pour la peau Classic | http://world-en.openbeautyfacts.org/product/20057060/creme-pour-la-peau-classic-cien
Lait corps hydratant nou

In [42]:
face_words = "face|facial|visage|skin-care|creme|cream|soin"
others = products[products["final_role"] == "other"]
face_others = others[others["categories_tags"].fillna("").str.contains(face_words, case=False)]
print("'Other' products with face-care words:", len(face_others))
miss_check = face_others.sample(min(20, len(face_others)), random_state=SEED)
for name, url in zip(miss_check["product_name"], miss_check["url"]):
    print(name, "|", url)

'Other' products with face-care words: 586
Crème réparatrice pieds très secs | http://world-en.openbeautyfacts.org/product/5000438001353/creme-reparatrice-pieds-tres-secs-scholl
Gel surgras bain et douche | http://world-en.openbeautyfacts.org/product/3596490000998/gel-surgras-bain-et-douche-roge-cavailles
men's Care club | http://world-en.openbeautyfacts.org/product/4056489318064/men-s-care-club-viens-men
Douche crème douceurs de nos régions à la vanille douce de Polynésie | http://world-en.openbeautyfacts.org/product/3600550560060/douche-creme-douceurs-de-nos-regions-a-la-vanille-douce-de-polynesie-dop
Dentifrice Protection Blancheur | http://world-en.openbeautyfacts.org/product/3517360018271/dentifrice-protection-blancheur-dentavie
Crème mains - peaux sèches | http://world-en.openbeautyfacts.org/product/3253581735411/creme-mains-peaux-seches-l-occitane
Dream Mat Mousse 48 Beige ensoleillé | http://world-en.openbeautyfacts.org/product/3600530171972/dream-mat-mousse-48-beige-ensoleille

**Missed products:** 20 random "other" products with face-care words to check how many are moisterizers or cleansers or sunscreen

In [ ]:
face_words = "face|facial|visage|skin-care|creme|cream|soin"
others = products[products["final_role"] == "other"]
face_others = others[others["categories_tags"].fillna("").str.contains(face_words, case=False)]
print("'Other' products with face-care words:", len(face_others))
face_others["main_category"].value_counts().head(15)

'Other' products with face-care words: 888


main_category
en:hand-creams                        142
fr:cremes-douche                       85
en:lip-cosmetics                       37
en:face-masks                          35
en:body-creams                         27
en:anti-wrinkles-creams                19
nl:beauty                              17
en:open-beauty-facts                   17
en:lip-balms                           17
fr:baumes-et-beurres-pour-le-corps     15
nl:skincare                            14
en:face-scrubs                         14
en:face-lotions                        13
ar:كريمات-مرطبة                        12
en:face                                12
Name: count, dtype: int64

## Split & Clean the Ingredients


In [43]:
def split_ingredients(text):
    """Turn an ingredient text into a clean list of ingredient names."""
    text = str(text).lower()
    text = text.replace("ingredients:", "").replace("ingrédients :", "").replace("ingrédients:", "")
    clean = []
    for piece in text.split(","):
        name = piece.strip().strip(".").strip()
        if name != "":
            clean.append(name)
    return clean

ALIASES = {
    "water": "aqua", "eau": "aqua", "aqua/water": "aqua", "water/aqua": "aqua",
    "aqua/water/eau": "aqua", "water/aqua/eau": "aqua",
    "fragrance": "parfum", "perfume": "parfum", "parfum/fragrance": "parfum",
    "glycerine": "glycerin", "glycérine": "glycerin",
    "vitamin e": "tocopherol",
}

def normalise_ingredient(name):
    """Return one standard name for an ingredient (no brackets, known synonyms merged)."""
    name = name.lower().split("(")[0]
    name = name.replace("*", "").strip().strip(".").strip()
    return ALIASES.get(name, name)

products["ingredient_list"] = products["ingredients_text"].apply(split_ingredients)
products["ingredient_list"] = products["ingredient_list"].apply(lambda lst: [normalise_ingredient(name) for name in lst])
products["nb_ingredients"] = products["ingredient_list"].apply(len)
products["ingredients_clean"] = products["ingredient_list"].apply(lambda lst: ", ".join(lst))
products[["product_name", "nb_ingredients", "ingredients_clean"]].head()

,product_name,nb_ingredients,ingredients_clean
5,Huile végétale Amande douce,1,prunus amygdalus dulcis oil
14,NaN,1,olea europaea fruit oil. 100% du total des ing...
15,Macérât huileux Lys,2,"helianthus annuus seed oil, lilium candidum fl..."
19,olé olé Aloé,7,"aqua, glycerin, hydroxyethyl cellulose, phenox..."
23,Base lavante,15,"aqua, lauryl glucoside, rosa damascena flower ..."


In [44]:
rows = []
for code, ingredient_list in zip(products["code"], products["ingredient_list"]):
    position = 1
    for name in ingredient_list:
        rows.append({"code": code, "position": position, "ingredient": name})
        position = position + 1

ingredients_long = pd.DataFrame(rows)
print(ingredients_long.shape)
ingredients_long["ingredient"].value_counts().head(30)

(121779, 3)


ingredient
aqua                      4754
parfum                    3945
glycerin                  3647
citric acid               2618
sodium benzoate           2406
limonene                  2238
linalool                  2189
sodium chloride           2029
sodium hydroxide          1664
sodium laureth sulfate    1335
cocamidopropyl betaine    1281
benzyl alcohol            1187
potassium sorbate         1116
tocopherol                1116
citronellol               1052
geraniol                   993
xanthan gum                989
propylene glycol           952
hexyl cinnamal             946
phenoxyethanol             880
coumarin                   746
benzyl salicylate          715
cetearyl alcohol           710
dimethicone                673
citral                     662
disodium edta              625
alcohol denat              594
caprylyl glycol            575
tocopheryl acetate         554
panthenol                  541
Name: count, dtype: int64

In [ ]:
((routine["is_light"] | routine["is_rich"]).groupby(routine["final_role"]).mean() * 100).round(1)

final_role
cleanser       25.3
moisturizer    78.0
sunscreen      28.8
dtype: float64

## The catalogue: one row per moisturizer, one column per questionnaire need


In [45]:
rows = []
for code, ingredient_list in zip(products["code"], products["ingredient_list"]):
    position = 1
    for name in ingredient_list:
        rows.append({"code": code, "position": position, "ingredient": name})
        position = position + 1

ingredients_long = pd.DataFrame(rows)
print(ingredients_long.shape)
ingredients_long.head(10)

(121779, 3)


,code,position,ingredient
0,00000311,1,prunus amygdalus dulcis oil
1,00000571,1,olea europaea fruit oil. 100% du total des ing...
2,00000587,1,helianthus annuus seed oil
3,00000587,2,lilium candidum flower extract
4,00000695,1,aqua
5,00000695,2,glycerin
6,00000695,3,hydroxyethyl cellulose
7,00000695,4,phenoxyethanol
8,00000695,5,aloe barbadensis leaf juice
9,00000695,6,allantoin


## COSING DATA CHECK

In [46]:
cosing = pd.read_csv("../raw_data/Swan_CosIng_Ingredient_Reference_2026-10-26.csv")
print(cosing.shape)
print(cosing.columns.tolist())
cosing.head()

(1521, 15)
['cosing_id', 'inci_name', 'functions', 'cas_numbers', 'ec_numbers', 'status', 'chemical_description', 'cosmetic_restriction', 'annex', 'reference_number', 'other_restrictions', 'sccs_opinions', 'sccs_urls', 'source_url', 'retrieved_date']


,cosing_id,inci_name,functions,cas_numbers,ec_numbers,status,chemical_description,cosmetic_restriction,annex,reference_number,other_restrictions,sccs_opinions,sccs_urls,source_url,retrieved_date
0,54170,"1,2-HEXANEDIOL",SKIN CONDITIONING | SOLVENT,6920-22-5,230-029-6,Active,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://ec.europa.eu/growth/tools-databases/co...,2026-10-26
1,54178,1-METHYLHYDANTOIN-2-IMIDE,SKIN CONDITIONING,60-27-5,200-466-7,Active,NaN,NaN,NaN,NaN,ban substances that deplete the ozone layer | ...,NaN,NaN,https://ec.europa.eu/growth/tools-databases/co...,2026-10-26
2,100236,2-MERCAPTONICOTINOYL GLYCINE,SKIN CONDITIONING - MISCELLANEOUS,2101538-28-5,NaN,Active,2-Mercaptonicotinoyl Glycine is the organic co...,NaN,NaN,NaN,NaN,NaN,NaN,https://ec.europa.eu/growth/tools-databases/co...,2026-10-26
3,89718,3-O-ETHYL ASCORBIC ACID,SKIN CONDITIONING,86404-04-8,NaN,Active,3-O-Ethyl Ascorbic Acid is the organic compound.,NaN,NaN,NaN,NaN,NaN,NaN,https://ec.europa.eu/growth/tools-databases/co...,2026-10-26
4,31529,4-METHYLBENZYLIDENE CAMPHOR,LIGHT STABILIZER | UV ABSORBER | UV FILTER,36861-47-9 / 38102-62-4,253-242-6 /,Active,4-Methylbenzylidene Camphor is the aromatic or...,II/1730,NaN,NaN,NaN,NaN,NaN,https://ec.europa.eu/growth/tools-databases/co...,2026-10-26


In [48]:
COSING_NAME_COLUMN = "..."       # e.g. the INCI name column
COSING_FUNCTION_COLUMN = "..."   # e.g. the function column

if COSING_NAME_COLUMN in cosing.columns and COSING_FUNCTION_COLUMN in cosing.columns:
    cosing["ingredient"] = cosing[COSING_NAME_COLUMN].astype(str).str.lower().str.strip()
    merged = ingredients_long.merge(cosing[["ingredient", COSING_FUNCTION_COLUMN]], on="ingredient", how="left")
    found = merged[COSING_FUNCTION_COLUMN].notnull()
    print("Ingredient mentions found in CosIng:", round(found.mean() * 100, 1), "%")
    print(merged[~found]["ingredient"].value_counts().head(20))
else:
    print("Fill COSING_NAME_COLUMN and COSING_FUNCTION_COLUMN with real column names from the output above.")

Fill COSING_NAME_COLUMN and COSING_FUNCTION_COLUMN with real column names from the output above.


In [50]:
def needs_referral(profile):
    """True if the answers mean we must refer the person instead of recommending products (T13)."""
    if profile["is_adult"] == False:
        return True
    elif profile["concern"] != "acne_prone":
        return True
    elif profile["blemish_level"] >= BLEMISH_REFERRAL_LEVEL:
        return True
    elif len(profile["referral_flags"]) > 0:
        return True
    else:
        return False

def eligible_products(profile, catalogue):
    """Products left after the 'avoid' filters (Q7)."""
    selected = catalogue
    if "fragrance" in profile["avoid"]:
        selected = selected[~selected["has_fragrance"]]
    if "essential_oils" in profile["avoid"]:
        selected = selected[~selected["has_essential_oil"]]
    return selected

def simple_score(profile, selected):
    """PLACEHOLDER score until Loubna's weighted scoring (T15)."""
    score = selected["has_acne_active"].astype(int)
    if profile["reactivity"] == True:
        score = score - selected["has_fragrance"].astype(int) - selected["has_essential_oil"].astype(int)
    if profile["skin_type"] in ["oily", "combination"]:
        score = score + selected["is_light"].astype(int)
    return score

def recommend(profile, catalogue):
    """Questionnaire answers -> 'referral', 'not enough products', or the top moisturizers."""
    if needs_referral(profile):
        return "referral"
    selected = eligible_products(profile, catalogue).copy()
    if len(selected) < NB_ROUTINE_PRODUCT:
        return "not enough products"          # never force a recommendation
    selected["score"] = simple_score(profile, selected)
    return selected.sort_values(["score", "completeness"], ascending=False).head(NB_ROUTINE_PRODUCT)

In [51]:
test_profiles = [
    {"name": "oily, reactive, avoid fragrance", "is_adult": True, "concern": "acne_prone", "blemish_level": 2,
     "referral_flags": [], "skin_type": "oily", "reactivity": True, "avoid": ["fragrance"]},
    {"name": "dry, not reactive, no avoid", "is_adult": True, "concern": "acne_prone", "blemish_level": 1,
     "referral_flags": [], "skin_type": "dry", "reactivity": False, "avoid": []},
    {"name": "avoid both", "is_adult": True, "concern": "acne_prone", "blemish_level": 3,
     "referral_flags": [], "skin_type": "combination", "reactivity": True, "avoid": ["fragrance", "essential_oils"]},
    {"name": "severe blemishes (referral)", "is_adult": True, "concern": "acne_prone", "blemish_level": 5,
     "referral_flags": [], "skin_type": "oily", "reactivity": False, "avoid": []},
    {"name": "pregnant (referral)", "is_adult": True, "concern": "acne_prone", "blemish_level": 2,
     "referral_flags": ["pregnant_or_breastfeeding"], "skin_type": "normal", "reactivity": False, "avoid": []},
]

for profile in test_profiles:
    result = recommend(profile, catalogue)
    print("====", profile["name"])
    if isinstance(result, str):
        print(result)
    else:
        print(result[["product_name", "brand", "score"]].to_string(index=False))

NameError: name 'catalogue' is not defined